In [1]:
from getpass import getpass
import os

REPLICATE_API_TOKEN = getpass()
os.environ["REPLICATE_API_TOKEN"] = "r8_YwrSiHNmeiOLZrFbDsh0iWGLmY55lmr3iQKPQ"

In [2]:
import torch as t
import numpy as np
import random
from transformer_lens import utils
import sys

import einops
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from tqdm import tqdm
from functools import partial
from datasets import load_dataset
from IPython.display import display

device = t.device("cuda" if t.cuda.is_available() else "cpu")

In [3]:
from nnsight import LanguageModel

model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [4]:
seed = 22

data = load_dataset("stas/openwebtext-10k", split="train")
tokenized_data = utils.tokenize_and_concatenate(data, model.tokenizer, max_length=128)
tokenized_data = tokenized_data.shuffle(seed)

In [5]:
#@title def topk
def unravel_index(flat_index, shape):

    indices = []
    for dim_size in reversed(shape):
        indices.append(flat_index % dim_size)
        flat_index = flat_index // dim_size
    return tuple(reversed(indices))

def topk(tensor, k):

    flat_tensor = tensor.flatten()

    top_values, flat_indices = t.topk(flat_tensor, k)

    original_indices = [unravel_index(idx.item(), tensor.size()) for idx in flat_indices]

    return top_values.tolist(), original_indices

In [6]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [7]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

local_dir = "../jbloom_dictionaries"

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir=local_dir)

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [8]:
def get_feature_acts(layer, sae_list=sae_list, num_batches=1000, minibatch_size=20):
    # get however many tokens we need
    toks = tokenized_data["tokens"][:num_batches]

    n_mini_batches = len(toks) // minibatch_size
    tok_batches = [toks[minibatch_size*i : minibatch_size*(i+1), :] for i in range(n_mini_batches)]

    feature_acts = None 

    for batch in tqdm(tok_batches):

        with model.trace(batch):
            activations = model.transformer.h[layer].input[0][0]

            middle = sae_list[layer](activations)

            acts = middle[1]
            acts.save()

        acts = acts.value.detach().cpu()

        if feature_acts is None:
            feature_acts = acts
        
        else:
            feature_acts = t.cat([feature_acts, acts], dim=0)

        del acts

    feature_acts[:, 0, :] = 0

    print("feature_acts has size:", feature_acts.size())

    return toks, feature_acts

In [9]:
layer = 10

TOKS, FEATURE_ACTS = get_feature_acts(layer, sae_list, num_batches=2_000, minibatch_size=200)

100%|██████████| 10/10 [00:34<00:00,  3.43s/it]

feature_acts has size: torch.Size([2000, 128, 24576])


In [9]:
from autointerp.single.explainer_prompts import get_explainer_template, get_simple_explainer_template

In [10]:
from dataclasses import dataclass
@dataclass
class ExplainerConfig:

    max_tokens : int = 2000
    temperature : float = 0.5

    batch_size: int = 10
    n_batches : int = 2
    runs_per_batch: int = 1

    left_context: int = 15
    right_context: int = 2
    activation_threshold: float = 0.4,

    verbose: bool = False

In [18]:
# delimiters
l = '<<'
r = '>>'

In [30]:
feature_id = 7000
left_context = 15
right_context = 2
threshold = 0.2

top_acts, top_inds = topk(FEATURE_ACTS[:,:, feature_id], 20)
max_act = top_acts[0]

top_examples_list = []

batch_size = TOKS.size(1)

for i, (batch, tok) in enumerate(top_inds):

    start = max(0, tok - left_context)
    end = min(batch_size, tok + right_context)

    example_toks = TOKS[batch, start:end]

    example_acts = FEATURE_ACTS[batch, start:end, feature_id]

    delimited_string = ''
    for pos in range(example_toks.size(0)):
        if example_acts[pos] > (threshold * max_act):
            delimited_string += l + model.tokenizer.decode(example_toks[pos]) + r
        else:
            delimited_string += model.tokenizer.decode(example_toks[pos])

446 102
400 29
607 106
1174 71
476 26
1535 71
1450 61
30 21
1413 55
1281 94
107 105
1775 78
607 107
568 65
1780 43
657 110
1412 76
15 38
871 122
1591 12


In [28]:
import replicate
class Explainer:
  def __init__(self, cfg):
    self.cfg = cfg
    self.n_examples = self.cfg.batch_size * self.cfg.n_batches

  def get_top_examples_list(self, feature_id):

    top_acts, top_inds = topk(FEATURE_ACTS[:,:, feature_id], self.n_examples)
    max_act = top_acts[0]

    top_examples_list = []
    for i in range(self.n_examples):
      start_pos = max(0, top_inds[i][1] - self.cfg.left_context)
      end_pos = min(TOKS.size(1), top_inds[i][1] + self.cfg.right_context + 1)
      example_toks = TOKS[top_inds[i][0], start_pos : end_pos]

      example_acts = FEATURE_ACTS[top_inds[i][0], start_pos : end_pos, feature_id]

      delimited_string = ''
      for pos in range(example_toks.size(0)):
        if example_acts[pos] > (self.cfg.activation_threshold * max_act):
          delimited_string += l + model.tokenizer.decode(example_toks[pos]) + r
        else:
          delimited_string += model.tokenizer.decode(example_toks[pos])

      delimited_string = delimited_string.replace(r+l, "")
      delimited_string = delimited_string.replace("{", "{{").replace("}", "}}")

      top_examples_list.append( delimited_string )

    self.top_examples_list = top_examples_list
    return top_examples_list


  def explain(self, feature_id):
    self.get_top_examples_list(feature_id)

    explanation_list = []

    for b in tqdm(range(self.cfg.n_batches)):

      # get batch of top examples, and convert to string
      examples_list = self.top_examples_list[b*self.cfg.batch_size : (b+1)*self.cfg.batch_size]
      examples_str = """"""
      for i in range(len(examples_list)):
        examples_str += "Example " + str(i+1) + ": " + examples_list[i] + "\n"

      if self.cfg.verbose: print(f"---------------BATCH {b}  TOP EXAMPLES----------------\n\n\n", examples_str, "n\n\n\n\n\n\n")


      for trial in range(self.cfg.runs_per_batch):
        if self.cfg.verbose: print(f"-------------------BATCH {b}  TRIAL {trial}-------------------\n\n\n")
        # prompt the explainer
        two_explanations = self.query_llm(examples_str)

        explanation_list += two_explanations

    return explanation_list


  def query_llm(self, examples_str):

    input = {
        "prompt": examples_str,
        "prompt_template": get_explainer_template(examples_str),
        "max_tokens" : 2000,
        "temperature" : self.cfg.temperature
    }
    output = replicate.run(
        "meta/meta-llama-3-70b-instruct",
        input=input
    )


    output_str = ''
    for i in output:
      output_str += i

    two_explanations = output_str.split("Step 4")[-1].split("1")[-1].split("2")  # this is janky as fuck, change this
    two_explanations = [e.strip(".): ") for e in two_explanations]

    if self.cfg.verbose: print(f"----------------EXPLAINER OUTPUT------------\n\n\n", output_str, "\n\n\n")

    return two_explanations

In [29]:
cfg = ExplainerConfig(
                      batch_size = 8,
                      n_batches = 2,
                      runs_per_batch = 3,
                      verbose = True,
                      temperature = 1.5,
                      activation_threshold = 0.2,
                      left_context = 15,
                      right_context = 4)

explainer = Explainer(cfg)

feature_id = 7000
long_explanation_list = explainer.get_top_examples_list(feature_id)

IndexError: list index out of range

In [21]:
long_explanation_list

[" places such as Nigeria. Even then security would not be perfect (and I<<'m not>> sure that airborne",
 ' it is merely testing and the shipments will be hitting full stride shortly. I<< am>> choosing to believe the',
 " Historically, it was proposed to get the 152mm M-51 and I<<'m not>> sure how W",
 ' DeepMind��s AI is surprisingly strong and getting stronger, but I<< am>> confident that I can',
 "tains superpowers. The comic hasn't even come out yet, but we<<'re>> guessing it's high",
 '�cool crew�� that hands out sunscreen. Honestly, I��<<m>> surprised this isn�',
 " as to place all the blame for these delays on Google's shoulders. We<<'re now>> four and a",
 ' lot of crosses into the area for Cristiano Ronaldo and Nani, I<< am not>> sure what they',
 " sources. Has the Till Report ever been published? My impression from what I<<'m>> reading here that it",
 ' meat, ��and it��s pretty good, I��<<m>> told,��',
 "th's screwdriver so this will be added to my DW collection! I<<'m>> goi